# Feature Engineering v2

## Objective 

This notebook extends the baseline feature set by incorporating spatial and contextual information to improve model performance.

The goal is:

    -capture player interactions.
    .introduce relative positioning features.
    -enhance spatial awerness of the model.
    -reduce prediction errors identified in the evaluation stage.

In [1]:
import sys 
from pathlib import Path 

import pandas as pd
import numpy as np 

sys.path.append(str(Path("..").resolve()))

from src.config import PROCESSED_DIR

In [2]:
input_path= PROCESSED_DIR/"baseline_train_v1.parquet"
df=pd.read_parquet(input_path)

print("Loaded dataset: ",df.shape)
display(df.head())

Loaded dataset:  (152305, 22)


,game_id,play_id,nfl_id,player_name,frame_id,x,y,s,a,dir,...,player_weight,player_height_inches,player_age,is_moving_right,player_position,player_side,player_role,ball_land_x,ball_land_y,num_frames_output
0,2023090700,101,46137,Justin Reid,1,51.32,20.69,0.31,0.49,79.43,...,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
1,2023090700,101,46137,Justin Reid,2,51.35,20.66,0.36,0.74,118.07,...,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
2,2023090700,101,46137,Justin Reid,3,51.39,20.63,0.44,0.76,130.89,...,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
3,2023090700,101,46137,Justin Reid,4,51.43,20.61,0.48,0.62,134.50,...,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
4,2023090700,101,46137,Justin Reid,5,51.48,20.58,0.54,0.44,129.79,...,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21


## Distance to Target

We compute the distance between player position and the predicted ball landing point.

In [3]:
df["dist_to_ball"]=np.sqrt(
    (df["x"]-df["ball_land_x"])**2 +
    (df["y"]-df["ball_land_y"])**2
)

In [4]:
df["speed_x"]=df["s"]*np.cos(np.deg2rad(df["dir"]))
df["speed_y"]=df["s"]*np.sin(np.deg2rad(df["dir"]))

In [5]:
df["acc_vector"]= df["a"]

In [6]:
df["x_centered"]=df["x"]-60 #field midpoint approx
df["y_centered"]=df["y"]-26.65

In [7]:
df["momentum"]=df["player_weight"]*df["s"]

In [8]:
df["speed_times_dir"]=df["s"]*df["dir"]
df["x_times_speed"]=df["x"]*df["s"]

In [14]:
df["speed_bin"] = pd.qcut(df["s"], q=5, duplicates="drop")
df["speed_bin_code"] = df["speed_bin"].cat.codes
df = df.drop(columns=["speed_bin"])

In [15]:
feature_cols_v2 = [
    "x", "y", "s", "a", "dir", "o",
    "absolute_yardline_number",
    "player_weight",
    "player_height_inches",
    "player_age",
    "is_moving_right",
    "player_position",
    "player_side",
    "player_role",

    # NEW FEATURES
    "dist_to_ball",
    "speed_x",
    "speed_y",
    "momentum",
    "x_centered",
    "y_centered"
]

In [16]:
output_path=PROCESSED_DIR/"baseline_train_v2.parquet"
df.to_parquet(output_path, index=False)

print("Saved v2 datase: ", output_path)

Saved v2 datase:  C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\data\processed\baseline_train_v2.parquet


## Summary

Feature Engineering v2 introduces spatial, directional, and interaction-based features.

These features aim to:

- provide the model with richer movement context,
- reduce prediction variance,
- and improve performance in complex scenarios identified in the evaluation stage.

The next step is to retrain the model using this enhanced feature set.